# Image Captioning — Train on Colab GPU

Runtime → Change runtime type → GPU (T4) before running any cell below.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## 1. Upload your project

Zip your local `Image captioning` folder (everything: `app.py`, `src/`, `requirements.txt`) and upload it here.

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import zipfile, os

zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall("/content/project")

%cd /content/project
!ls

If the zip extracted into a subfolder (e.g. `/content/project/Image captioning/`), `cd` into that instead:

```python
%cd "/content/project/Image captioning"
```

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt
!python -m spacy download en_core_web_sm -q

## 3. Get the Flickr8k dataset

Upload your `kaggle.json` API token (Kaggle account → Settings → Create New API Token).

In [ ]:
from google.colab import files
kaggle_token = files.upload()  # select kaggle.json

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle
!kaggle datasets download -d adityajn105/flickr8k
!unzip -q flickr8k.zip -d flickr8k

In [ ]:
!ls flickr8k

Check the output above. If the images ended up in `flickr8k/Images` vs `flickr8k/images` (case matters on Linux, unlike Windows), update `root_folder` in `src/training/train.py` to match — or run the rename cell below.

In [ ]:
import os

if os.path.isdir("flickr8k/Images") and not os.path.isdir("flickr8k/images"):
    os.rename("flickr8k/Images", "flickr8k/images")

print(os.listdir("flickr8k"))

## 4. Set epochs and train

Bump `num_epochs` back up now that you have a GPU — e.g. 20–30 is a reasonable start for Flickr8k.

In [ ]:
!sed -i 's/num_epochs = 2/num_epochs = 20/' src/training/train.py
!grep num_epochs src/training/train.py

In [ ]:
!python -m src.training.train

## 5. Download the trained checkpoint

Once training finishes (or you Ctrl+C the cell once loss looks good), download `my_checkpoint.pth.tar` and drop it into your local `Image captioning` folder.

In [ ]:
from google.colab import files
files.download("my_checkpoint.pth.tar")

## (Optional) Watch loss live

Run this in a cell above your training cell, before starting training, to see the TensorBoard loss curve update live.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/flickr